In [ ]:
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.nonparametric import kaplan_meier_estimator
from sksurv.compare import compare_survival

from sklearn import set_config
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
set_config(display="text")  # displays text representation of estimators

import warnings
from sklearn.exceptions import FitFailedWarning
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", FitFailedWarning)

In [ ]:
import anndata as ad
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import math
from scipy import stats
import scipy
from tqdm import tqdm
import os
from matplotlib import rc
from sklearn import mixture
from sklearn import preprocessing
import matplotlib
import matplotlib.ticker as tkr
from statannotations.Annotator import Annotator
from datetime import date
import plotly.express as px
import plotly.graph_objects as go
from matplotlib.pyplot import cm
from scipy.cluster import hierarchy
import matplotlib as mpl
import gseapy as gp
from scipy.stats import chi2_contingency

from matplotlib.patches import Patch
import random
import matplotlib.colors as mcolors
from scipy.stats import zscore

import sklearn
from sklearn.preprocessing import MinMaxScaler

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
sc.settings.set_figure_params(dpi=200, facecolor='white')
plt.rcParams['axes.grid'] = False

In [ ]:
print('\n'.join(f'{m.__name__}=={m.__version__}' for m in globals().values() if getattr(m, '__version__', None)))

In [ ]:
os.makedirs('output',exist_ok=True)

# Clinical data loading

In [ ]:
clinical_data_df = pd.read_excel('data/clinical_data.xlsx',header=0, index_col=0)
clinical_data_df

In [ ]:
clinical_data_prepared_df = clinical_data_df.copy()

# Group IPSS-M categories in 2 groups: Lower IPSS-M (VL+L+ML) and Higher IPSS-M (MH+H+VH)
clinical_data_prepared_df['ipss_m_cat_merged'] = 'NA'

VL_L_ML_index = clinical_data_prepared_df.loc[(clinical_data_prepared_df['ipss_m_cat']=='VL') | (clinical_data_prepared_df['ipss_m_cat']=='L') | (clinical_data_prepared_df['ipss_m_cat']=='ML')].index
MH_H_VH_index = clinical_data_prepared_df.loc[(clinical_data_prepared_df['ipss_m_cat']=='MH') | (clinical_data_prepared_df['ipss_m_cat']=='H') | (clinical_data_prepared_df['ipss_m_cat']=='VH')].index
clinical_data_prepared_df['ipss_m_cat_merged'][VL_L_ML_index] = 'Lower_IPSSM'
clinical_data_prepared_df['ipss_m_cat_merged'][MH_H_VH_index] = 'Higher_IPSSM'

# Identifying dated HSCT events
followup_idx = clinical_data_prepared_df['date_of_hsct'].notnull()

# Censor date and survival status at HSCT
clinical_data_prepared_df['date_last_follow_up'][followup_idx] = clinical_data_prepared_df['date_of_hsct'][followup_idx]
clinical_data_prepared_df['alive_0_dead_1'][followup_idx] = 0

# Transforming alive dead status to boolean
clinical_data_prepared_df.rename(columns={"alive_0_dead_1": "Status"},inplace=True)
clinical_data_prepared_df['Status'] = clinical_data_prepared_df['Status'].astype('bool')

# Calculating survival time from MDS diagnostic to last follow-up
first_diagnostic_list = list(clinical_data_prepared_df['date_of_mds_diagnosis'])
last_diagnostic_list = list(clinical_data_prepared_df['date_last_follow_up'])
survival_list = []
survival_list_months = []
survival_list_years = []

for i,j in zip(first_diagnostic_list,last_diagnostic_list):
    delay = j - i
    survival_list.append(delay)
    if not pd.isnull(j) and not pd.isnull(j): # check for not NaT
        delay_months = int(((j - i)/np.timedelta64(1, 'M')))
        survival_list_months.append(delay_months)
        delay_years = float(((j - i)/np.timedelta64(1, 'Y')))
        survival_list_years.append(delay_years)
    else:
        survival_list_months.append(pd.NaT)
        survival_list_years.append(pd.NaT)

        
clinical_data_prepared_df['Survival_in_days'] = survival_list
clinical_data_prepared_df['Survival_in_days'] = clinical_data_prepared_df['Survival_in_days'].dt.days

clinical_data_prepared_df['Survival_in_months'] = survival_list_months
clinical_data_prepared_df['Survival_in_months'] = clinical_data_prepared_df['Survival_in_months']

clinical_data_prepared_df['Survival_in_years'] = survival_list_years
clinical_data_prepared_df['Survival_in_years'] = clinical_data_prepared_df['Survival_in_years']

In [ ]:
clinical_data_prepared_df

# German data loading

In [ ]:
Discovery_rna = pd.read_csv('data/Discovery_cohort_bulk_RNA_seq.txt',sep='\t',index_col=0)
Discovery_rna

In [ ]:
Discovery_rna_log = np.log1p(Discovery_rna).copy()
Discovery_rna_log

# Calculating ssGSEA scores

In [ ]:
gmt_dict = {}
# with open('data/MDS_BM_Single_cell_celltypes_markers_corrected.gmt','rU') as infile:
with open('data/Immune_cell_types_genesets.gmt','rU') as infile:
    for line in infile:
        tmp = line.split()
        gmt_dict[tmp[0]] = tmp[2:]
GMT_full = list(gmt_dict.keys())

# Create list of immune only cell-types and remove progenitor cells
GMT_immune_only = [e for e in GMT_full if e not in ('GMP_scbm','HSC_scbm','LMPP_scbm','Plasmablast_scbm','Prog_RBC_scbm','Prog_Mk_scbm','pDC_scbm','Prog_B_1_scbm','Prog_B_2_scbm','Prog_DC_scbm')]

In [ ]:
ssgsea_scores = gp.ssgsea(data=Discovery_rna_log,
               gene_sets='data/Immune_cell_types_genesets.gmt',
               outdir=None,
               sample_norm_method='rank',
               no_plot=True,
               min_size=2)

In [ ]:
# Conversion of ssGSEA result object to dataframe
ssGSEA_res = ssgsea_scores.res2d

Discovery_ssGSEA_df = pd.DataFrame(0, index=Discovery_rna_log.columns.values, columns=ssGSEA_res.Term.unique())

for result in range(ssGSEA_res.shape[0]):
    patient = ssGSEA_res.loc[result]['Name']
    cell_type = ssGSEA_res.loc[result]['Term']
    nes = ssGSEA_res.loc[result]['NES']
    es = ssGSEA_res.loc[result]['ES']
    Discovery_ssGSEA_df[cell_type][patient] = es # select es for Raw Enrichment Scores or nes for Normalised Enrichment Scores

# Discovery_ssGSEA_df.to_csv('output/Discovery_cohort_ssGSEA_scores.txt',sep='\t',index=True)
Discovery_ssGSEA_df

# Visualising ssGSEA scores

In [ ]:
Output_folder = "output/ssGSEA/"
os.makedirs(Output_folder,exist_ok=True)

plt.figure(figsize=(12,8))

sorted_order = ['CD14_Mono_scbm','CD16_Mono_scbm','CD4_Memory_scbm','CD4_Naive_scbm','CD8_Effector_1_scbm','CD8_Effector_2_scbm','CD8_Memory_1_scbm','CD8_Memory_2_scbm','CD8_Naive_scbm','cDC2_scbm','gdT_scbm','Memory_B_scbm','Naive_B_scbm','CD56_bright_NK_scbm','NK_scbm','MAIT_scbm','Treg_scbm','Naive_B']
ax = Discovery_ssGSEA_df.boxplot(column=list(Discovery_ssGSEA_df[sorted_order].columns),rot=85)

ax.set_yticklabels(ax.get_ymajorticklabels(), fontsize = 14, style='normal',weight='bold')
ax.set_xticklabels(ax.get_xmajorticklabels(), fontsize = 14, style='normal',weight='bold')

ax.grid(False)
plt.tight_layout()
plt.savefig(Output_folder+'Discovery_cohort_Immune_only_ssGSEA_ES_boxplots.png', dpi=200)
plt.savefig(Output_folder+'Discovery_cohort_Immune_only_ssGSEA_ES_boxplots.svg')
plt.show()
plt.close()

# Cox-LASSO modelling with Cross-Validation

In [ ]:
def plot_coefficients(coefs, n_highlight):
    _, ax = plt.subplots(figsize=(9, 6))
    n_features = coefs.shape[0]
    alphas = coefs.columns
    for row in coefs.itertuples():
        ax.semilogx(alphas, row[1:], ".-", label=row.Index)

    alpha_min = alphas.min()
    top_coefs = coefs.loc[:, alpha_min].map(abs).sort_values().tail(n_highlight)
    for name in top_coefs.index:
        coef = coefs.loc[name, alpha_min]
        plt.text(alpha_min, coef, name + "   ", horizontalalignment="right", verticalalignment="center")

    ax.yaxis.set_label_position("right")
    ax.yaxis.tick_right()
    ax.grid(True)
    ax.set_xlabel("alpha")
    ax.set_ylabel("coefficient")

In [ ]:
def CoxLasso_modelling_with_CV(output_folder=None,input_data=None,input_sign=None,annot_data=None,cv_type=None,fold_num=5,l1_ratio=1):
    
    ### Create model output folder
    os.makedirs(output_folder,exist_ok=True)

    input_data_df = input_data.copy() # German_BM_ssGSEA_df_low_subset, German_BM_ssGSEA_df_high_subset
    input_signatures = input_sign # GMT_immune_order, GMT_sergio_order, GMT_sergio_exhaust_order, GMT_order
    annot_data_df = annot_data.copy()
    
    ### Creation of the input data for CoxLasso
    cohort_annot = pd.merge(annot_data_df, input_data_df,left_index=True,right_index=True).copy()

    cohort_annot_y = cohort_annot[['Status','Survival_in_months']].copy()
    v = cohort_annot_y.reset_index(drop=True)
    cohort_annot_y = np.rec.fromrecords(v, names=v.columns.tolist())

    X = cohort_annot[input_signatures].copy()
    y = cohort_annot_y.copy()
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    ### Running simple CoxLasso
    cox_lasso = CoxnetSurvivalAnalysis(l1_ratio=l1_ratio, alpha_min_ratio='auto',normalize=False)
    cox_lasso.fit(X_scaled, y)

    coefficients_lasso = pd.DataFrame(cox_lasso.coef_, index=X.columns, columns=np.round(cox_lasso.alphas_, 6))
    plot_coefficients(coefficients_lasso, n_highlight=5)

    plt.tight_layout()
    plt.savefig(output_folder+'CoxLasso_coefficients_pointplot.png', dpi=300)
    plt.savefig(output_folder+'CoxLasso_coefficients_pointplot.svg', dpi=300)
#     plt.show()
    plt.close()

    ### Optimising alpha for CoxLasso via cross-validation
    coxnet_pipe = make_pipeline(StandardScaler(), CoxnetSurvivalAnalysis(l1_ratio=l1_ratio, alpha_min_ratio='auto', max_iter=10000))
    coxnet_pipe.fit(X, y)

    estimated_alphas = coxnet_pipe.named_steps["coxnetsurvivalanalysis"].alphas_
    if cv_type == 'kfold_norm':
        cv = KFold(n_splits=fold_num, shuffle=False)
    elif cv_type == 'kfold_shuffle':
        cv = KFold(n_splits=fold_num, shuffle=True, random_state=0) # fix seed for reproducibility
    elif cv_type == 'kfold_repeat':
        cv = RepeatedKFold(n_splits=fold_num, n_repeats=5, random_state=0) # fix seed for reproducibility
    else:
        print('No such Kfold type...')
        return
    gcv = GridSearchCV(
        make_pipeline(StandardScaler(), CoxnetSurvivalAnalysis(l1_ratio=l1_ratio)),
        param_grid={"coxnetsurvivalanalysis__alphas": [[v] for v in estimated_alphas]},
        cv=cv,
        n_jobs=4,
        return_train_score=True
    ).fit(X, y)

    cv_results = pd.DataFrame(gcv.cv_results_)

    alphas = cv_results.param_coxnetsurvivalanalysis__alphas.map(lambda x: x[0])
    mean = cv_results.mean_test_score
    std = cv_results.std_test_score

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(alphas, mean)
    ax.fill_between(alphas, mean - std, mean + std, alpha=0.15)
    ax.set_xscale("log")
    ax.set_ylabel("concordance index")
    ax.set_xlabel("alpha")
    ax.axvline(gcv.best_params_["coxnetsurvivalanalysis__alphas"][0], c="C1")
    ax.axhline(0.5, color="grey", linestyle="--")
    ax.grid(True)

    plt.tight_layout()
    plt.savefig(output_folder+'CoxLasso_alpha_optimisation.png', dpi=300)
    plt.savefig(output_folder+'CoxLasso_alpha_optimisation.svg', dpi=300)
    # plt.show()
    plt.close()

    ### Exporting optimised model
    best_model = gcv.best_estimator_.named_steps["coxnetsurvivalanalysis"]
    best_coefs = pd.DataFrame(best_model.coef_, index=X.columns, columns=["coefficient"])
    best_coefs.to_csv(output_folder+'CoxLasso_best_fit_coefs.txt',sep='\t',index=True)

    non_zero = np.sum(best_coefs.iloc[:, 0] != 0)

    non_zero_coefs = best_coefs.query("coefficient != 0")
    coef_order = non_zero_coefs.abs().sort_values("coefficient").index

    _, ax = plt.subplots(figsize=(6, 8))
    non_zero_coefs.loc[coef_order].plot.barh(ax=ax, legend=False)
    ax.set_xlabel("coefficient")
    ax.grid(True)

    plt.tight_layout()
    plt.savefig(output_folder+'CoxLasso_coef_forestplot.png', dpi=300)
    plt.savefig(output_folder+'CoxLasso_coef_forestplot.svg', dpi=300)
    # plt.show()
    plt.close()

    ### Saving best result from fit
    pd.DataFrame.from_dict(gcv.cv_results_).iloc[gcv.best_index_].to_csv(output_folder+'CoxLasso_best_fit_result.txt',sep='\t',index=True)
    pd.DataFrame.from_dict(gcv.cv_results_).to_csv(output_folder+'CoxLasso_full_fit_result.txt',sep='\t',index=True)

    return

In [ ]:
Kfold_type = 'kfold_norm' # Available algorithm for CV: kfold_norm, kfold_shuffle, kfold_repeat 
Kfold_size = 10 # Size of k-fold for CV
L1_ratio = 1 # L1/L2 ratio for CoxNet penalisation
output_folder = 'output/CoxLasso_modelling_'+str(Kfold_size)+'_'+Kfold_type+'_CV/' # Output folder of CoxLasso modelling results
tests_dict = {}
tests_dict['Whole_cohort_Immune_only'] = {'output':output_folder,
                                          'input_d':Discovery_ssGSEA_df,
                                          'input_s':GMT_immune_only,
                                          'input_a':clinical_data_prepared_df}

for test in tqdm(tests_dict):
    CoxLasso_modelling_with_CV(output_folder = tests_dict[test]['output'],
                               input_data = tests_dict[test]['input_d'],
                               input_sign = tests_dict[test]['input_s'],
                               annot_data = tests_dict[test]['input_a'],
                               cv_type=Kfold_type,fold_num=Kfold_size,l1_ratio=L1_ratio)


# MAD based immune risk groups

In [ ]:
Output_risk_score_folder = output_folder+"Risk_score/"
os.makedirs(Output_risk_score_folder,exist_ok=True)

### Loading the CoxLasso coefficients and calculating the risk score
whole_immune_coef = pd.read_csv(output_folder+'CoxLasso_best_fit_coefs.txt',sep='\t',index_col=0)

# Rescale the ssGSEA scores as done during modelling
scaler = StandardScaler()
Discovery_ssGSEA_scalled_df = pd.DataFrame(scaler.fit_transform(Discovery_ssGSEA_df),
                                                        columns=Discovery_ssGSEA_df.columns,
                                                        index=Discovery_ssGSEA_df.index)

Discovery_ssGSEA_coef_df = Discovery_ssGSEA_scalled_df.copy()
Discovery_ssGSEA_coef_df = Discovery_ssGSEA_coef_df[whole_immune_coef.index]
for signature in Discovery_ssGSEA_coef_df.columns.values:
    Discovery_ssGSEA_coef_df[signature] = Discovery_ssGSEA_coef_df[signature] * whole_immune_coef.T[signature][0]

Discovery_ssGSEA_coef_df['Risk_score'] = Discovery_ssGSEA_coef_df.sum(axis=1)

### Plotting the immune risk score
sns.displot(data=Discovery_ssGSEA_coef_df, x='Risk_score', kind="kde",
           height=6, aspect=12/6, color='k')

plt.axvline(x=Discovery_ssGSEA_coef_df['Risk_score'].median(),ls='--', lw=2, color='r')

### Plotting the Median Absolute Deviation lines, used to separate the immune risk categories
mad = scipy.stats.median_abs_deviation(Discovery_ssGSEA_coef_df['Risk_score'])
mad_coef = 1

plt.axvline(x=Discovery_ssGSEA_coef_df['Risk_score'].median()-(mad_coef*mad),ls='--', lw=2, color='k')
plt.axvline(x=Discovery_ssGSEA_coef_df['Risk_score'].median()+(mad_coef*mad),ls='--', lw=2, color='k')

plt.tight_layout()
plt.savefig(Output_risk_score_folder+'Discovery_3_MAD_categories_risk_score_distplot.png', dpi=200)
plt.savefig(Output_risk_score_folder+'Discovery_3_MAD_categories_risk_score_distplot.svg', dpi=200)
plt.show()
plt.close()

In [ ]:
### Creation of output folder for figures
Output_survival_folder = output_folder+"Survival_curves/"
os.makedirs(Output_survival_folder,exist_ok=True)

### Loading of CoxLasso coefficients to generate the immune risk (IR) score
whole_immune_coef = pd.read_csv(output_folder+'CoxLasso_best_fit_coefs.txt',sep='\t',index_col=0)

# Rescale the ssGSEA scores as done during modelling
scaler = StandardScaler()
Discovery_ssGSEA_scalled_df = pd.DataFrame(scaler.fit_transform(Discovery_ssGSEA_df),
                                                        columns=Discovery_ssGSEA_df.columns,
                                                        index=Discovery_ssGSEA_df.index)

Discovery_ssGSEA_coef_df = Discovery_ssGSEA_scalled_df.copy()
Discovery_ssGSEA_coef_df = Discovery_ssGSEA_coef_df[whole_immune_coef.index]
for signature in Discovery_ssGSEA_coef_df.columns.values:
    Discovery_ssGSEA_coef_df[signature] = Discovery_ssGSEA_coef_df[signature] * whole_immune_coef.T[signature][0]

Discovery_ssGSEA_coef_df['Risk_score'] = Discovery_ssGSEA_coef_df.sum(axis=1)

comp_name = 'Risk_score'

num_cat = 'Risk_score_3_cat'

Discovery_ssGSEA_coef_df[num_cat] = 'NA'

Discovery_ssGSEA_coef_df = Discovery_ssGSEA_coef_df.sort_values(by='Risk_score', ascending=True)

### Calculating the Median Absolute Deviation (MAD) to separate patients into 3 risk categories
mad = scipy.stats.median_abs_deviation(Discovery_ssGSEA_coef_df['Risk_score'])
mad_coef = 1

Low_immune_risk_idx = Discovery_ssGSEA_coef_df.loc[Discovery_ssGSEA_coef_df['Risk_score'] <= Discovery_ssGSEA_coef_df['Risk_score'].median()-(mad_coef*mad)].index
Medium_immune_risk_idx = Discovery_ssGSEA_coef_df.loc[(Discovery_ssGSEA_coef_df['Risk_score'] > Discovery_ssGSEA_coef_df['Risk_score'].median()-(mad_coef*mad)) & (Discovery_ssGSEA_coef_df['Risk_score'] < Discovery_ssGSEA_coef_df['Risk_score'].median()+(mad_coef*mad))].index
High_immune_risk_idx = Discovery_ssGSEA_coef_df.loc[Discovery_ssGSEA_coef_df['Risk_score'] >= Discovery_ssGSEA_coef_df['Risk_score'].median()+(mad_coef*mad)].index

Discovery_ssGSEA_coef_df[num_cat][Low_immune_risk_idx] = 'Low_immune_risk'
Discovery_ssGSEA_coef_df[num_cat][Medium_immune_risk_idx] = 'Medium_immune_risk'
Discovery_ssGSEA_coef_df[num_cat][High_immune_risk_idx] = 'High_immune_risk'

Discovery_cohort_IR_groups = pd.merge(clinical_data_prepared_df, Discovery_ssGSEA_coef_df,left_index=True,right_index=True).copy()

Discovery_cohort_IR_groups_subset = Discovery_cohort_IR_groups[['ipss_m_cat_merged',num_cat]].copy()

Discovery_cohort_IR_groups_subset['Patient'] = Discovery_cohort_IR_groups_subset.index

### Plotting survival curves of IR groups in whole cohort
cat_name_list = ['Low_immune_risk', 'Medium_immune_risk', 'High_immune_risk']
risk_color = {'Low_immune_risk':'#7FC5E2',
              'Medium_immune_risk':'#E4BE95',
              'High_immune_risk':'#B3B3B3'}

tmp_annot_merged = pd.merge(clinical_data_prepared_df, Discovery_cohort_IR_groups_subset,left_index=True,right_index=True).copy()

tmp_annot_merged_y = tmp_annot_merged[['Status','Survival_in_months']].copy()
v = tmp_annot_merged_y.reset_index(drop=True)
tmp_annot_merged_y = np.rec.fromrecords(v, names=v.columns.tolist())

annot_BM_merged_plot = tmp_annot_merged.copy()

plt.figure(figsize=(8, 5))

for value in cat_name_list:

    mask = annot_BM_merged_plot[num_cat] == value

    time_cell, survival_prob_cell, conf_int = kaplan_meier_estimator(
        annot_BM_merged_plot["Status"][mask], annot_BM_merged_plot["Survival_in_months"][mask], conf_type="log-log"
    )
    plt.step(time_cell, survival_prob_cell, where="post", label=f"{value} (n = {mask.sum()})",
            color=risk_color[value])
    plt.fill_between(time_cell, conf_int[0], conf_int[1], alpha=0.25, step="post",color=risk_color[value])

plt.ylim(0, 1)
plt.ylabel("est. probability of survival $\hat{S}(t)$")
plt.xlabel("time $t$ in months")
plt.legend(loc="best")

group_indicator = annot_BM_merged_plot.loc[:, num_cat]
tmp_annot_merged_y_tmp = tmp_annot_merged_y[group_indicator != 'NA'].copy()

group_indicator = group_indicator.loc[group_indicator != 'NA']
chisq, pvalue, stats, covar = compare_survival(
        tmp_annot_merged_y_tmp, group_indicator, return_stats=True)

plt.title('Discovery whole cohort IR groups\n pval='+str(pvalue))
plt.tight_layout()
plt.savefig(Output_survival_folder+'Discovery_whole_cohort_3_IR_groups_survival_curve.png', dpi=200)
plt.savefig(Output_survival_folder+'Discovery_whole_cohort_3_IR_groups_survival_curve.svg', dpi=200)
plt.show()
plt.close()

### Subseting the cohort to Lower IPSS-M category patients
Lower_IPSSM_Low_immune_risk_idx = Discovery_cohort_IR_groups_subset.loc[(Discovery_cohort_IR_groups_subset[num_cat] == 'Low_immune_risk') & (Discovery_cohort_IR_groups_subset['ipss_m_cat_merged'] == 'Lower_IPSSM')].index
Lower_IPSSM_Medium_immune_risk_idx = Discovery_cohort_IR_groups_subset.loc[(Discovery_cohort_IR_groups_subset[num_cat] == 'Medium_immune_risk') & (Discovery_cohort_IR_groups_subset['ipss_m_cat_merged'] == 'Lower_IPSSM')].index
Lower_IPSSM_High_immune_risk_idx = Discovery_cohort_IR_groups_subset.loc[(Discovery_cohort_IR_groups_subset[num_cat] == 'High_immune_risk') & (Discovery_cohort_IR_groups_subset['ipss_m_cat_merged'] == 'Lower_IPSSM')].index

Discovery_cohort_IR_groups_subset_Lower_IPSSM = Discovery_cohort_IR_groups_subset.loc[Lower_IPSSM_Low_immune_risk_idx.append(Lower_IPSSM_Medium_immune_risk_idx).append(Lower_IPSSM_High_immune_risk_idx)].copy()

Discovery_cohort_IR_groups_subset_Lower_IPSSM['IPSSM_with_risk_score'] = 'NA'
Discovery_cohort_IR_groups_subset_Lower_IPSSM['IPSSM_with_risk_score'][Lower_IPSSM_Low_immune_risk_idx] = 'Lower_IPSSM_Low_immune_risk'
Discovery_cohort_IR_groups_subset_Lower_IPSSM['IPSSM_with_risk_score'][Lower_IPSSM_Medium_immune_risk_idx] = 'Lower_IPSSM_Medium_immune_risk'
Discovery_cohort_IR_groups_subset_Lower_IPSSM['IPSSM_with_risk_score'][Lower_IPSSM_High_immune_risk_idx] = 'Lower_IPSSM_High_immune_risk'

### Plotting survival curves of IR groups in Lower IPSS-M
num_cat_loc = 'IPSSM_with_risk_score'

cat_name_list = ['Lower_IPSSM_Low_immune_risk', 'Lower_IPSSM_Medium_immune_risk', 'Lower_IPSSM_High_immune_risk']
risk_color = {'Lower_IPSSM_Low_immune_risk':'#425EFE',
              'Lower_IPSSM_Medium_immune_risk':'#3F8E57',
              'Lower_IPSSM_High_immune_risk':'#F73449'}

tmp_annot_merged = pd.merge(clinical_data_prepared_df, Discovery_cohort_IR_groups_subset_Lower_IPSSM,left_index=True,right_index=True).copy()

tmp_annot_merged_y = tmp_annot_merged[['Status','Survival_in_months']].copy()
v = tmp_annot_merged_y.reset_index(drop=True)
tmp_annot_merged_y = np.rec.fromrecords(v, names=v.columns.tolist())

annot_BM_merged_plot = tmp_annot_merged.copy()

plt.figure(figsize=(8, 5))

for value in cat_name_list:

    mask = annot_BM_merged_plot[num_cat_loc] == value

    time_cell, survival_prob_cell, conf_int = kaplan_meier_estimator(
        annot_BM_merged_plot["Status"][mask], annot_BM_merged_plot["Survival_in_months"][mask], conf_type="log-log"
    )
    plt.step(time_cell, survival_prob_cell, where="post", label=f"{value} (n = {mask.sum()})",
            color=risk_color[value])
    plt.fill_between(time_cell, conf_int[0], conf_int[1], alpha=0.25, step="post",color=risk_color[value])

plt.ylim(0, 1)
plt.ylabel("est. probability of survival $\hat{S}(t)$")
plt.xlabel("time $t$ in months")
plt.legend(loc="best")

group_indicator = annot_BM_merged_plot.loc[:, num_cat_loc]
tmp_annot_merged_y_tmp = tmp_annot_merged_y[group_indicator != 'NA'].copy()

group_indicator = group_indicator.loc[group_indicator != 'NA']
chisq, pvalue, stats, covar = compare_survival(
        tmp_annot_merged_y_tmp, group_indicator, return_stats=True)

plt.title('Discovery cohort Lower IPSS-M IR groups\n pval='+str(pvalue))
plt.tight_layout()
plt.savefig(Output_survival_folder+'Discovery_cohort_Lower_IPSSM_3_IR_groups_survival_curve.png', dpi=200)
plt.savefig(Output_survival_folder+'Discovery_cohort_Lower_IPSSM_3_IR_groups_survival_curve.svg', dpi=200)
plt.show()
plt.close()

### Subseting the cohort to Lower IPSS-M category patients
Higher_IPSSM_Low_immune_risk_idx = Discovery_cohort_IR_groups_subset.loc[(Discovery_cohort_IR_groups_subset[num_cat] == 'Low_immune_risk') & (Discovery_cohort_IR_groups_subset['ipss_m_cat_merged'] == 'Higher_IPSSM')].index
Higher_IPSSM_Medium_immune_risk_idx = Discovery_cohort_IR_groups_subset.loc[(Discovery_cohort_IR_groups_subset[num_cat] == 'Medium_immune_risk') & (Discovery_cohort_IR_groups_subset['ipss_m_cat_merged'] == 'Higher_IPSSM')].index
Higher_IPSSM_High_immune_risk_idx = Discovery_cohort_IR_groups_subset.loc[(Discovery_cohort_IR_groups_subset[num_cat] == 'High_immune_risk') & (Discovery_cohort_IR_groups_subset['ipss_m_cat_merged'] == 'Higher_IPSSM')].index

Discovery_cohort_IR_groups_subset_Higher_IPSSM = Discovery_cohort_IR_groups_subset.loc[Higher_IPSSM_Low_immune_risk_idx.append(Higher_IPSSM_Medium_immune_risk_idx).append(Higher_IPSSM_High_immune_risk_idx)].copy()

Discovery_cohort_IR_groups_subset_Higher_IPSSM['IPSSM_with_risk_score'] = 'NA'
Discovery_cohort_IR_groups_subset_Higher_IPSSM['IPSSM_with_risk_score'][Higher_IPSSM_Low_immune_risk_idx] = 'Higher_IPSSM_Low_immune_risk'
Discovery_cohort_IR_groups_subset_Higher_IPSSM['IPSSM_with_risk_score'][Higher_IPSSM_Medium_immune_risk_idx] = 'Higher_IPSSM_Medium_immune_risk'
Discovery_cohort_IR_groups_subset_Higher_IPSSM['IPSSM_with_risk_score'][Higher_IPSSM_High_immune_risk_idx] = 'Higher_IPSSM_High_immune_risk'

num_cat_loc = 'IPSSM_with_risk_score'

cat_name_list = ['Higher_IPSSM_Low_immune_risk', 'Higher_IPSSM_Medium_immune_risk', 'Higher_IPSSM_High_immune_risk']
risk_color = {'Higher_IPSSM_Low_immune_risk':'#425EFE',
              'Higher_IPSSM_Medium_immune_risk':'#3F8E57',
              'Higher_IPSSM_High_immune_risk':'#F73449'}

tmp_annot_merged = pd.merge(clinical_data_prepared_df, Discovery_cohort_IR_groups_subset_Higher_IPSSM,left_index=True,right_index=True).copy()

tmp_annot_merged_y = tmp_annot_merged[['Status','Survival_in_months']].copy()
v = tmp_annot_merged_y.reset_index(drop=True)
tmp_annot_merged_y = np.rec.fromrecords(v, names=v.columns.tolist())

annot_BM_merged_plot = tmp_annot_merged.copy()

plt.figure(figsize=(8, 5))

for value in cat_name_list:

    mask = annot_BM_merged_plot[num_cat_loc] == value

    time_cell, survival_prob_cell, conf_int = kaplan_meier_estimator(
        annot_BM_merged_plot["Status"][mask], annot_BM_merged_plot["Survival_in_months"][mask], conf_type="log-log"
    )
    plt.step(time_cell, survival_prob_cell, where="post", label=f"{value} (n = {mask.sum()})",
            color=risk_color[value])
    plt.fill_between(time_cell, conf_int[0], conf_int[1], alpha=0.25, step="post",color=risk_color[value])

plt.ylim(0, 1)
plt.ylabel("est. probability of survival $\hat{S}(t)$")
plt.xlabel("time $t$ in months")
plt.legend(loc="best")

group_indicator = annot_BM_merged_plot.loc[:, num_cat_loc]
tmp_annot_merged_y_tmp = tmp_annot_merged_y[group_indicator != 'NA'].copy()

group_indicator = group_indicator.loc[group_indicator != 'NA']
chisq, pvalue, stats, covar = compare_survival(
        tmp_annot_merged_y_tmp, group_indicator, return_stats=True)

plt.title('Discovery cohort Higher IPSS-M IR groups\n pval='+str(pvalue))
plt.tight_layout()
plt.savefig(Output_survival_folder+'Discovery_cohort_Higher_IPSSM_3_IR_groups_survival_curve.png', dpi=200)
plt.savefig(Output_survival_folder+'Discovery_cohort_Higher_IPSSM_3_IR_groups_survival_curve.svg', dpi=200)
plt.show()
plt.close()

# Exporting risk scores and IR groups annotations

In [ ]:
Output_IR_groups_folder = output_folder+"Tables/"
os.makedirs(Output_IR_groups_folder,exist_ok=True)

Discovery_ssGSEA_coef_df['Risk_score'].to_csv(Output_IR_groups_folder+'Discovery_cohort_risk_scores_whole_cohort.txt',sep='\t',index=True)

Discovery_cohort_IR_groups_subset.drop('Patient',axis=1).to_csv(Output_IR_groups_folder+'Discovery_cohort_IR_groups_whole_cohort.txt',sep='\t',index=True)